<a href="https://colab.research.google.com/github/Amol24D/LLMPlayground/blob/main/llm_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 1: Build an LLM Playground

Welcome to your first project! In this project, you'll build a simple large language model (LLM) playground, an interactive environment where you can experiment with LLMs and understand how they work under the hood.

The goal here is to understand the foundations and mechanics behind LLMs rather than relying on higher-level abstractions or frameworks. You'll see what happens under the hood, how an LLM receives text, processes it, and generates a response. In later projects, you'll use frameworks like `Ollama` and `LangChain` that simplify many of these steps. But before that, this project will help you build a solid mental model of how LLMs actually work.

---
## Environment Setup
We'll use Google Colab, a free browser-based platform that lets you run Python code and machine learning models without installing anything locally. Go to [Google Colab](https://colab.research.google.com/) and upload this notebook to get started.

If you prefer to run the project locally, you need a reproducible setup. Open a terminal in the same directory as this notebook and run the environment setup commands below to install dependencies and create an isolated environment.

### Step 1: Use Conda or uv to install the project dependencies.

#### Option 1: Conda

[Conda](https://docs.conda.io/projects/conda/en/latest/user-guide/index.html) is an open-source package and environment manager that lets you create isolated environments and install Python and non-Python dependencies together.

```bash
# Create and activate the conda environment
conda env create -f environment.yaml && conda activate llm_playground
```

#### Option 2: uv (faster)

[uv](https://docs.astral.sh/uv/) (faster) is a fast Python package installer and virtual environment tool written in Rust that aims to replace pip, pip-tools, and virtualenv with a single, high-performance workflow.

```bash
# Install uv (skip if already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create venv and install dependencies
uv venv .venv --python 3.11 && source .venv/bin/activate
uv pip install -r requirements.txt
```

### Step 2: Register this environment as a Jupyter kernel
This step is optional. Do it only if your environment doesn’t appear in Jupyter’s kernel list.
```bash
python -m ipykernel install --user --name=llm_playground --display-name "llm_playground"
```

Now switch the kernel to `llm_playground` (Kernel → Change Kernel).

---
## Learning Objectives  
- Understand tokenization and how raw text is converted into a sequence of discrete tokens
- Load and inspect a pretrained LLM (GPT-2) using Hugging Face
- Understand logits, probabilities, and how models predict the next token
- Count and explore model parameters to understand model scale
- Explore decoding strategies: greedy decoding and top-p (nucleus) sampling
- See the leap from GPT-2 (simple text completion) to a modern model that understands questions and thinks before answering
- Look ahead: inference engines for serving models in practice


Let's get started!

---

In [30]:
# Confirm required libraries are installed and working.
import torch, transformers, tiktoken
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("✅ Environment check complete. You're good to go!")

torch 2.10.0+cu128 | transformers 5.0.0
✅ Environment check complete. You're good to go!


# 1: Tokenization

A neural network cannot process raw text directly. It needs numbers.
Tokenization is the process of converting text into numerical IDs that models can understand. In this section, you will learn how tokenization works in practice and why it is an essential step in every language model pipeline.

Tokenization methods generally fall into three main categories:
1. Word-level
2. Character-level
3. Subword-level

### 1.1 - Word-level tokenization
This method splits text by whitespace and treats each word as a single token. In the next cell, you will implement a basic word-level tokenizer by building a vocabulary that maps words to IDs and writing `encode` and `decode` functions.

In [31]:
# Creating a tiny corpus. In practice, a corpus is generally the entire internet-scale dataset used for training.

corpus = [
    "Please do not deploy on friday",
    "it works on my machine",
    "extra spicy noodles are a bad idea!",
    "tokens are tiny pieces of text",
]

# Step 1: Build vocabulary (all unique words in the corpus) and mappings
vocab = []
word2id = {}
id2word = {}

# Collect all unique words
import re
# Iterate over every sentence in the corpus
for sentence in corpus:
    # Split sentence into individual words on whitespace
    for word in sentence.split():
        # Strip punctuation (e.g. "idea!" -> "idea"), keep alphanumerics only
        word = re.sub(r'[^\w]', '', word)

        # Only add the word if it's non-empty and not already in the vocab
        if word and word not in word2id:
            word_id = len(vocab)        # next available integer ID
            vocab.append(word)          # add to ordered vocab list
            word2id[word] = word_id     # forward mapping: word -> ID
            id2word[word_id] = word     # reverse mapping: ID -> word

print(f"Vocab size: {len(vocab)}")
print(f"vocab:    {vocab}")
print(f"word2id:  {word2id}")
print(f"id2word:  {id2word}")


Vocab size: 22
vocab:    ['Please', 'do', 'not', 'deploy', 'on', 'friday', 'it', 'works', 'my', 'machine', 'extra', 'spicy', 'noodles', 'are', 'a', 'bad', 'idea', 'tokens', 'tiny', 'pieces', 'of', 'text']
word2id:  {'Please': 0, 'do': 1, 'not': 2, 'deploy': 3, 'on': 4, 'friday': 5, 'it': 6, 'works': 7, 'my': 8, 'machine': 9, 'extra': 10, 'spicy': 11, 'noodles': 12, 'are': 13, 'a': 14, 'bad': 15, 'idea': 16, 'tokens': 17, 'tiny': 18, 'pieces': 19, 'of': 20, 'text': 21}
id2word:  {0: 'Please', 1: 'do', 2: 'not', 3: 'deploy', 4: 'on', 5: 'friday', 6: 'it', 7: 'works', 8: 'my', 9: 'machine', 10: 'extra', 11: 'spicy', 12: 'noodles', 13: 'are', 14: 'a', 15: 'bad', 16: 'idea', 17: 'tokens', 18: 'tiny', 19: 'pieces', 20: 'of', 21: 'text'}


In [32]:
# Step 2: Define encode and decode functions
def encode(text):
    # converts text to token IDs
    # split the input text into words and look up each word's ID in word2id
    return [word2id[word] for word in text.split()]


def decode(ids):
    # converts token IDs back to text
    # look up each ID in id2word and join the resulting words with spaces
    return " ".join([id2word[id] for id in ids])


In [33]:
# Step 3: Test your tokenizer with some random sentences.
# Try a sentence with unseen words and see what happens (and how to fix it)

# Test 1: sentence that exists in the corpus — should work fine
print(encode("it works on my machine"))   # [6, 7, 4, 8, 9]
print(decode([6, 7, 4, 8, 9]))            # it works on my machine

# Test 2: sentence with words from different corpus sentences — should still work
print(encode("tokens are tiny pieces of text"))  # [17, 13, 18, 19, 20, 21]

# Test 3: sentence with an unseen word — will raise a KeyError
try:
    print(encode("do not deploy on monday"))  # "monday" is not in vocab
except KeyError as e:
    print(f"KeyError: {e} is not in the vocabulary")

# Fix: handle unseen words with a special <UNK> token
UNK = "<UNK>"                        # unknown token placeholder
word2id[UNK] = len(vocab)            # assign it the next available ID
id2word[len(vocab)] = UNK            # add reverse mapping
vocab.append(UNK)                    # add to vocab list

def encode_safe(text):
    # like encode, but replaces any unseen word with the <UNK> token ID
    return [word2id.get(word, word2id[UNK]) for word in text.split()]

# Test 4: same unseen sentence, now handled gracefully
print(encode_safe("do not deploy on monday"))   # "monday" maps to <UNK> ID
print(decode(encode_safe("do not deploy on monday")))  # "do not deploy on <UNK>"

[6, 7, 4, 8, 9]
it works on my machine
[17, 13, 18, 19, 20, 21]
KeyError: 'monday' is not in the vocabulary
[1, 2, 3, 4, 22]
do not deploy on <UNK>


While word-level tokenization is simple and easy to understand, it has two key limitations that make it impractical for large-scale models:
1. Large vocabulary size: every new word or variation (for example, run, runs, running) increases the total vocabulary, leading to higher memory and training costs.
2. Out-of-vocabulary (OOV) problem: the model cannot handle unseen or rare words that were not part of the training vocabulary, so they must be replaced with a generic [UNK] token.

The next section introduces character-level tokenization, where text is represented as individual characters instead of words.

### 1.2 - Character-level tokenization

In this approach, every single character (including spaces, punctuation, and even emojis) is assigned its own ID.

In the next cell, we will rebuild a tokenizer using the same corpus as before, but this time with a character-level approach.
For simplicity, we will only use English letters (a-z, A-Z) and punctuation.

In [34]:
import string

# Step 1: Create a vocabulary that includes uppercase letters, lowercase letters, and punctuation.
vocab = []
char2id = {}
id2char = {}

# string.ascii_letters = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
# string.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
# combine all characters we want in the vocabulary into one iterable
for char in string.ascii_letters + string.punctuation+' ':
    char_id = len(vocab)       # next available integer ID
    vocab.append(char)         # add character to ordered vocab list
    char2id[char] = char_id    # forward mapping: char -> ID
    id2char[char_id] = char    # reverse mapping: ID -> char

In [35]:
# Step 2: Implement encode() and decode() functions to convert between text and IDs.
def encode(text):
    # convert text to list of IDs
    # iterate over each character in the text and look up its ID in char2id
    return [char2id[char] for char in text]


def decode(ids):
    # Convert list of IDs to text
    # look up each ID in id2char and join characters directly (no spaces — chars already include spacing)
    return "".join([id2char[id] for id in ids])

In [36]:
# Step 3: Test your tokenizer on some sample words.

# Test 1: encode a simple word and check the IDs
print(encode("Hello"))        # [33, 4, 11, 11, 14] — each char maps to its ID

# Test 2: decode the IDs back to confirm round-trip works
print(decode(encode("Hello")))  # Hello

# Test 3: try punctuation since we included it in the vocab
print(encode("Wait!"))        # IDs for W, a, i, t, !
print(decode(encode("Wait!"))) # Wait!

# Test 4: try an unseen character like a space or digit to see the KeyError
try:
    print(encode("Hi there"))
    print(decode(encode("Hi there")))
except KeyError as e:
    print(f"KeyError: {e} is not in the vocabulary")

[33, 4, 11, 11, 14]
Hello
[48, 0, 8, 19, 52]
Wait!
[33, 8, 84, 19, 7, 4, 17, 4]
Hi there


Character-level tokenization solves the out-of-vocabulary problem but introduces new challenges:

1. Longer sequences: because each word becomes many tokens, models need to process much longer inputs.
2. Weaker semantic representation: individual characters carry very little meaning, so models must learn relationships across many steps.
3. Higher computational cost: longer sequences lead to more tokens per input, which increases training and inference time.

To find a better balance between vocabulary size and sequence length, we move to subword-level tokenization next.

### 1.3 - Subword-level tokenization

Subword methods such as `Byte-Pair Encoding (BPE)`, `WordPiece`, and `SentencePiece` **learn** common groups of characters and merge them into tokens. For example, the word **unbelievable** might turn into three tokens: **["un", "believ", "able"]**. This approach strikes a balance between word-level and character-level methods and fixes their limitations.

The BPE algorithm builds a vocabulary iteratively using the following process:
1. Start with individual characters or bytes (each character is a token).
2. Count all adjacent pairs of tokens in a large text corpus.
3. Merge the most frequent pair into new tokens.

Repeat steps 2 and 3 until you reach the desired vocabulary size (for example, 50,000 tokens).

In the next cell, you will experiment with BPE in practice to see how it compresses text into meaningful subword units. Instead of implementing the algorithm from scratch, you will use a pretrained tokenizer, which was already trained on a large text corpus to build its vocabulary, such as the data used to train `GPT-2`. This allows you to see how BPE works in practice with a real, learned vocabulary.

In [37]:
from transformers import AutoTokenizer

# Step 1: Load a pretrained GPT-2 tokenizer from Hugging Face.
# Refer to this to learn more: https://huggingface.co/docs/transformers/en/model_doc/gpt2

# AutoTokenizer.from_pretrained downloads and caches the GPT-2 tokenizer config and vocab files
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [38]:
# Step 2: Encode a sample sentence and inspect how BPE breaks it into subword tokens.
# Hint: bpe_tok has encode, decode, and convert_ids_to_tokens methods.

sentence = "unnecessarily spicy noodles is a unmanageable idea !"

# encode() converts the sentence into a list of integer token IDs
ids = tokenizer.encode(sentence)
print("Token IDs:     ", ids)

# convert_ids_to_tokens() shows the actual BPE subword pieces each ID maps to
tokens = tokenizer.convert_ids_to_tokens(ids)
print("Subword tokens:", tokens)

# decode() converts the token IDs back to the original string
decoded = tokenizer.decode(ids)
print("Decoded text:  ", decoded)

Token IDs:      [403, 10789, 3093, 26880, 31103, 318, 257, 29389, 496, 540, 2126, 5145]
Subword tokens: ['un', 'necess', 'arily', 'Ġspicy', 'Ġnoodles', 'Ġis', 'Ġa', 'Ġunman', 'age', 'able', 'Ġidea', 'Ġ!']
Decoded text:   unnecessarily spicy noodles is a unmanageable idea !


### 1.4 - TikToken

`tiktoken` is a fast, production-ready library for tokenization used by OpenAI models.
It is designed for efficiency and consistency with how OpenAI counts tokens in GPT models.

In this section, you will explore how different model families use different tokenizers. We will compare tokenizers used to train `GPT-2` and more powerful models such as `GPT-4`. By trying both, you will see how tokenization has evolved to handle more diverse text (including emojis and special characters) while remaining efficient.

In the next cell, you will use tiktoken to load these encodings and inspect how each one splits the same text. You may find reading this doc helpful: https://github.com/openai/tiktoken

In [39]:
import tiktoken

# Compare GPT-2 and GPT-4 tokenizers using tiktoken.

# Step 1: Load two tokenizers
# encoding_for_model picks the correct BPE vocab for each model
enc_gpt2 = tiktoken.encoding_for_model("gpt2")
enc_gpt4 = tiktoken.encoding_for_model("gpt-4")

# Step 2: Encode the same sentence with both and observe how they differ
sentences = [
    "I 💀 at this codebase",           # common emoji
    "🚀🚀🚀 to the moon!",              # repeated emoji
    "GPT-4o is 10x better than GPT-3", # numbers + hyphens + model names
    "naïve café résumé",               # accented characters
    "C++ and Python3.11 are 🔥",       # code-like tokens
    "😂😭🤯🥹",                         # emoji-only, no text
    "1000000 vs 1_000_000",            # number formatting
    "http://openai.com/gpt-4",         # URL
]

for s in sentences:
    ids_gpt2 = enc_gpt2.encode(s)
    ids_gpt4 = enc_gpt4.encode(s)
    decoded = enc_gpt4.decode(ids_gpt4)
    print(f"decoded:       {decoded}")
    #print(f"Text:       {s}")
    print(f"GPT-2 ({len(ids_gpt2)} tokens): {[enc_gpt2.decode_single_token_bytes(i) for i in ids_gpt2]}")
    print(f"GPT-4 ({len(ids_gpt4)} tokens): {[enc_gpt4.decode_single_token_bytes(i) for i in ids_gpt4]}")
    print()

# decode_single_token_bytes() shows exactly what bytes each token represents
tokens_gpt2 = [enc_gpt2.decode_single_token_bytes(id) for id in ids_gpt2]
tokens_gpt4 = [enc_gpt4.decode_single_token_bytes(id) for id in ids_gpt4]

print("GPT-2 tokens:       ", tokens_gpt2)
print("GPT-4 tokens:       ", tokens_gpt4)

# compare token counts — GPT-4 generally uses fewer tokens (larger vocab = better compression)
print("GPT-2 token count:  ", len(ids_gpt2))
print("GPT-4 token count:  ", len(ids_gpt4))

# Step 3: Inspect special tokens used in each tokenizer
# special_tokens_set contains reserved tokens like <|endoftext|>, <|im_start|> etc.
print("GPT-2 special tokens:", enc_gpt2.special_tokens_set)
print("GPT-4 special tokens:", enc_gpt4.special_tokens_set)

# n_vocab shows total vocabulary size — GPT-4 has a significantly larger vocab than GPT-2
print("GPT-2 vocab size:   ", enc_gpt2.n_vocab)
print("GPT-4 vocab size:   ", enc_gpt4.n_vocab)

decoded:       I 💀 at this codebase
GPT-2 (8 tokens): [b'I', b' \xf0\x9f', b'\x92', b'\x80', b' at', b' this', b' code', b'base']
GPT-4 (7 tokens): [b'I', b' \xf0\x9f\x92', b'\x80', b' at', b' this', b' code', b'base']

decoded:       🚀🚀🚀 to the moon!
GPT-2 (13 tokens): [b'\xf0\x9f', b'\x9a', b'\x80', b'\xf0\x9f', b'\x9a', b'\x80', b'\xf0\x9f', b'\x9a', b'\x80', b' to', b' the', b' moon', b'!']
GPT-4 (13 tokens): [b'\xf0\x9f', b'\x9a', b'\x80', b'\xf0\x9f', b'\x9a', b'\x80', b'\xf0\x9f', b'\x9a', b'\x80', b' to', b' the', b' moon', b'!']

decoded:       GPT-4o is 10x better than GPT-3
GPT-2 (14 tokens): [b'G', b'PT', b'-', b'4', b'o', b' is', b' 10', b'x', b' better', b' than', b' G', b'PT', b'-', b'3']
GPT-4 (15 tokens): [b'G', b'PT', b'-', b'4', b'o', b' is', b' ', b'10', b'x', b' better', b' than', b' G', b'PT', b'-', b'3']

decoded:       naïve café résumé
GPT-2 (6 tokens): [b'na', b'\xc3\xafve', b' caf\xc3\xa9', b' r\xc3\xa9', b'sum', b'\xc3\xa9']
GPT-4 (7 tokens): [b'na', b'\xc3\

Try changing the input sentence and observe how different tokenizers behave.
Experiment with:
- Emojis, special characters, or punctuation
- Code snippets or structured text
- Non-English text (for example, Japanese, French, or Arabic)

If you are curious, you can also attempt to implement the BPE algorithm yourself using a small text corpus to see how token merges are learned in practice.

### 1.5 - Key Takeaways
- **Word-level**: simple and intuitive, but limited by large vocabularies and out-of-vocabulary issues
- **Character-level**: flexible and covers all text, but produces long sequences that are harder to model
- **Subword / BPE**: balances both worlds and is the default choice for most modern LLMs
- **TikToken**: a production-ready tokenizer used in OpenAI models, demonstrating how optimized subword vocabularies are applied in real systems

# 2: What is a Language Model?

At its core, a **language model** is a function that predicts the next token. Given a sequence of tokens `[t₁, t₂, …, tₙ]`, it outputs probabilities for the next token `tₙ₊₁`.

Models like GPT-2 use many stacked Transformer layers. Each layer mixes information between tokens (attention) and transforms it (feed-forward). Together, these layers learn patterns in text. At the end, the model produces logits: one score per token in the vocabulary. Higher logits mean the token is more likely to be next.

In this section, you’ll load GPT-2, look at its architecture, count its parameters, and see how it predicts the next token.

### 2.1: Loading GPT-2

There are different ways to load and run pretrained language models.

In this project, we’ll use Hugging Face Transformers, a popular Python library that makes it easy to download models like GPT-2 and run them locally.

There are also dedicated inference engines for serving and running modern LLMs more efficiently, such as **Ollama**, **SGLang**, and **vLLM**. We’ll explore those in future projects.

In the next cell, you’ll load GPT-2 and inspect its architecture.

In [40]:
import torch
from transformers import GPT2LMHeadModel

# Step 1: Load the smallest GPT-2 model (124M parameters) using the Hugging Face transformers library.
# from_pretrained downloads and caches the model weights from Hugging Face Hub
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Step 2: Print the full model architecture
# printing the model object shows every layer and its shape in a readable tree
print(model)

# Step 3: Inspect the first Transformer block by printing its layers
# model.transformer.h is a list of all transformer blocks; index 0 = first block
print(model.transformer.h[0])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)
GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, ele

### 2.2: Counting Parameters

You often hear that an LLM has a few million or billion parameters. But what does that actually mean? Every weight and bias value inside every layer is a parameter. These are the numbers that the model learned during training.

Next, you will count the total number of parameters in GPT-2 and break them down by component to see where most of the model's capacity lives.

In [41]:
# Count the total number of parameters in GPT-2
# Hint: use model.parameters() and the .numel() method on each parameter tensor

# model.parameters() yields every weight tensor in the model
# .numel() returns the total number of elements in a tensor
# summing across all tensors gives the total parameter count
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")  # should be ~124 million

Total parameters: 124,439,808


**Think about scale:** GPT-2 Small has 124M parameters. GPT-4 is estimated to have over 1 trillion. If each parameter is a 16-bit floating point number (2 bytes), how much memory would you need just to store GPT-2 in RAM? What about a 70B parameter model?

### 2.3: From Text to Predictions

When you pass a sequence of tokens through a language model, it produces a tensor of logits with shape
`(batch_size, seq_len, vocab_size)`.
Each position in the sequence receives a vector of scores representing how likely every possible token is to appear next. By applying a softmax function on the last dimension, these logits can be converted into probabilities that sum to 1.

In the next cell, you will feed a sentence into GPT-2, print the shape of its logits, and display the five most likely next tokens predicted for the final position in the sequence.

In [42]:
from transformers import AutoTokenizer

# Step 1: Load the GPT-2 tokenizer (the model is already loaded above)
# from_pretrained loads the same tokenizer that was used during GPT-2 training
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [43]:
# Step 2: Tokenize input text
text = "Hello my name is"

# tokenizer() converts text to a dict with input_ids (token IDs) as a tensor, ready for the model
inputs = tokenizer(text, return_tensors="pt")  # "pt" = PyTorch tensors

print(inputs)          # shows the full dict: input_ids and attention_mask
print(inputs.input_ids) # just the token ID tensor

{'input_ids': tensor([[15496,   616,  1438,   318]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
tensor([[15496,   616,  1438,   318]])


In [44]:
# Step 3: Pass the input IDs to the model

# torch.no_grad() disables gradient computation — we're just doing inference, not training
with torch.no_grad():
    # pass input_ids to the model; outputs contains logits and optionally hidden states
    outputs = model(**inputs)

# logits are the raw unnormalized scores for every token in the vocab at each position
# shape: [batch_size, seq_len, vocab_size] = [1, 3, 50257]
print(outputs.logits.shape)

torch.Size([1, 4, 50257])


In [45]:
import torch.nn.functional as F

# Step 4: Predict the next token
# Take the logits from the final position, apply softmax to get probabilities,
# and then extract the top 5 most likely next tokens.

# grab logits at the last token position [-1] — this predicts what comes after "name"
# shape goes from [1, 3, 50257] -> [50257]
last_logits = outputs.logits[0, -1, :]

# softmax converts raw logits into probabilities that sum to 1
probs = F.softmax(last_logits, dim=-1)

# topk returns the 5 highest probabilities and their corresponding token IDs
top_probs, top_ids = torch.topk(probs, 5)

# decode each token ID back to a readable string and print alongside its probability
for token_id, prob in zip(top_ids, top_probs):
    token_str = tokenizer.decode(token_id)
    print(f"{token_str!r:20s} {prob:.4f}")

' John'              0.0102
' L'                 0.0074
' J'                 0.0068
' K'                 0.0068
' James'             0.0067


### 2.4: Key Takeaway

A language model is not a black box or something mysterious.
It is a large composition of simple, understandable layers such as attention and feed-forward networks, trained together to predict the next token in a sequence.

By learning this next-token prediction task at scale, the model gradually develops an internal understanding of language structure, meaning, and context, which allows it to generate coherent and relevant text.

# 3: Text Generation (Decoding)
Once a language model can predict the next-token probabilities, we can use it to generate text. This is called text generation or decoding.

Conceptually, generation is a loop:

1. Feed the current token sequence into the model.

2. The model outputs a probability distribution over the next token.

3. A decoding algorithm picks the next token from that distribution.

4. Append the chosen token to the sequence and repeat.

In practice, libraries like Hugging Face Transformers provide generate() methods that handle this loop for you, including stopping conditions, batching, and efficiency tricks.

Different decoding strategies control how the next token is chosen, and how creative or deterministic the output feels:

- **Greedy decoding**: always pick the token with the highest probability. Simple and consistent, but often repetitive.

- **Top-p (nucleus) sampling**: randomly sample from the smallest set of tokens whose cumulative probability exceeds a threshold p. This adds variety while keeping outputs coherent.

In the following sections, you'll use generate() to produce text from GPT-2 using both greedy decoding and top-p sampling.

### 3.1: Greedy Decoding
In this section, you will use GPT-2 and Hugging Face's built-in generate method to produce text using the greedy decoding strategy.

In [46]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2"

# Step 1. Load GPT-2 model and tokenizer.
# AutoModelForCausalLM is the correct class for text generation (causal = predicts next token)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# Step 2. tokenize the prompt, then use model.generate() to generate text, then print the text
prompt = "Hi my name"

# tokenize the prompt into input_ids tensor, "pt" = PyTorch
inputs = tokenizer(prompt, return_tensors="pt")

# model.generate() autoregressively predicts one token at a time up to max_new_tokens
output_ids = model.generate(
    **inputs,
    max_new_tokens=20,   # how many tokens to generate beyond the prompt
    do_sample=False,     # greedy decoding — always picks the highest probability token
)

# decode the full output tensor back to a human-readable string
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hi my name is John, and I'm a writer and a writer. I'm a writer and a writer.


In [47]:
# Step 3: Write a helper function that wraps model.generate for greedy decoding,
# then test it on multiple prompts to observe repetition patterns.

tests = ["Once upon a time", "What is 2+2?", "Suggest a party theme."]

def generate(model, tokenizer, prompt, max_new_tokens=32):
    # tokenize the prompt into a PyTorch tensor
    inputs = tokenizer(prompt, return_tensors="pt")

    # greedy decoding — deterministically picks the highest probability token at each step
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,  # number of tokens to generate beyond the prompt
        do_sample=False,                # no sampling — pure greedy
    )

    # decode the output back to text, skipping any special tokens like <|endoftext|>
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

for prompt in tests:
    print(f"\n GPT-2 | Greedy")
    print(generate(model, tokenizer, prompt, 80))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 GPT-2 | Greedy


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time, the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and

 GPT-2 | Greedy


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is 2+2?

2+2 is the number of times you can use a spell to cast a spell.

2+2 is the number of times you can use a spell to cast a spell.

2+2 is the number of times you can use a spell to cast a spell.

2+2 is the number of times you can use a spell to cast a spell.

 GPT-2 | Greedy
Suggest a party theme.

The party theme is a simple, simple, and fun way to get your friends to join you.

The party theme is a simple, simple, and fun way to get your friends to join you. The party theme is a simple, simple, and fun way to get your friends to join you. The party theme is a simple, simple, and fun way to get your friends


Naively selecting the single most probable token at each step (known as greedy decoding) often leads to poor results in practice:
- Repetition loops: phrases like "The cat is is is…"
- Short-sighted choices: the most likely token right now might lead to incoherent text later

These issues are why more advanced decoding methods such as top-p (nucleus) sampling are commonly used to make model outputs more diverse and natural.

### 3.2: Top-p (Nucleus) Sampling
Top-p sampling (also called nucleus sampling) introduces controlled randomness into text generation. Instead of always picking the single most likely token, it samples from the smallest set of tokens whose cumulative probability exceeds a threshold `p` (e.g., 0.9).

This allows the model to explore multiple plausible continuations, producing more diverse and natural-sounding text while still staying coherent.

In this section, you will implement a generate function that supports both greedy and top-p strategies.

In [48]:
# Use model.generate to use top-p instead of greedy
prompt = "Hi my name"

# tokenize the prompt into a PyTorch tensor
inputs = tokenizer(prompt, return_tensors="pt")

# top-p (nucleus) sampling — at each step, sample only from the smallest set of tokens
# whose cumulative probability exceeds p (here 0.9), then renormalize and sample from that set
output_ids = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,       # must be True to enable any sampling strategy
    top_p=0.6,            # nucleus — only consider tokens covering 90% of the probability mass
    temperature=1.0,      # no scaling — keep the original distribution shape
)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hi my name is Ryan Boulware. I am a freelance writer and the owner of the website "Flexible Living". I am a self-taught living, creative, and self-employed person.


In [49]:
# Write a helper function that wraps model.generate for top-p decoding,
# then test it on multiple prompts
def generate(model, tokenizer, prompt, strategy="top_p", max_new_tokens=128):
    # tokenize the prompt into a PyTorch tensor
    inputs = tokenizer(prompt, return_tensors="pt")

    # build generation kwargs based on the chosen strategy
    if strategy == "greedy":
        # greedy — always picks the single highest probability token, no randomness
        strategy_kwargs = {
            "do_sample": False,
        }
    elif strategy == "top_p":
        # nucleus sampling — sample from tokens covering 90% of probability mass
        strategy_kwargs = {
            "do_sample": True,
            "top_p": 0.9,
            "temperature": 1.0,
        }
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Use 'greedy' or 'top_p'.")

    # generate tokens using the selected strategy
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,  # number of tokens to generate beyond the prompt
        **strategy_kwargs,              # unpack whichever strategy kwargs were built above
    )

    # decode the full output back to a readable string
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


tests = ["Once upon a time", "What is 2+2?", "Suggest a party theme."]
for prompt in tests:
    print(f"\n== GPT-2 | Top-p ==")
    print(generate(model, tokenizer, prompt, "top_p", 32))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



== GPT-2 | Top-p ==


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time when people had no idea about the impact of such events, we had the idea to develop the concept of an anti-nuclear group. We were not in any

== GPT-2 | Top-p ==


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is 2+2?

There are several basic arguments, but most of them make sense for one reason: When comparing the cost of a given item and the cost of a new

== GPT-2 | Top-p ==
Suggest a party theme. This one can be found in the section "How to Add a Party Theme".

The party theme and description will be found in the section "How to


# 4: From Simple GPT2 to Modern LLMs

So far, we have used **GPT-2**, one of the earliest publicly available language models (released in 2019, 124M parameters). GPT-2 can only do one thing: **complete text**. Given some input, it predicts what words come next. It has no concept of questions, instructions, or conversation. If you type `"What is 2+2?"`, GPT-2 will continue the text as if it were part of a web page or article. It does not understand you are asking a question.

Modern language models are a different. Models like **Qwen3-0.6B** (2025, 600M parameters) have gone through additional training stages that unlock fundamentally new capabilities:

- **Instruction following**: they interpret your input as a request and produce a helpful response
- **Conversation**: they maintain context across a multi-turn dialogue
- **Reasoning**: they can *think step-by-step* before answering, using internal reasoning tokens (`<think>...</think>`) to work through a problem before giving a final answer

In this section, you will see this dramatic contrast firsthand by giving the same prompts to both models. In week 4, we'll learn about thinking and reasoning models in detail.

### 4.1: Chat Templates

Instruction-tuned models expect input in a structured **chat format**, not raw text. Instead of receiving `"What is 2+2?"` as plain text, the model expects a formatted message like:

```
<|user|>What is 2+2?<|assistant|>
```

Each model family defines its own format. The Hugging Face `tokenizer.apply_chat_template()` method handles this formatting automatically. Without it, even an instruction-tuned model receives unstructured text and falls back to simple completion behavior.

### 4.2: GPT-2 vs. Qwen3-0.6B

In the next cells, you will load both models and feed the same prompt to each one:

- **GPT-2**: receives the raw prompt and blindly continues the text
- **Qwen3-0.6B**: receives a properly formatted chat message, *thinks* through the problem, and produces a direct answer

In [50]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load both GPT-2 and Qwen/Qwen3-0.6B models using HuggingFace `.from_pretrained` method.

# --- GPT-2 (124M parameters) ---
# smaller, older model — good baseline for comparison
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")

# --- Qwen3-0.6B (600M parameters) ---
# newer, larger model — should produce noticeably better output quality
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
qwen_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.float16,   # float16 halves memory usage vs float32 — important for larger models
    device_map="auto",           # automatically places layers on available hardware (GPU if present, else CPU)
)

print("GPT-2 loaded:      ", sum(p.numel() for p in gpt2_model.parameters()), "parameters")
print("Qwen3-0.6B loaded: ", sum(p.numel() for p in qwen_model.parameters()), "parameters")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


GPT-2 loaded:       124439808 parameters
Qwen3-0.6B loaded:  751632384 parameters


Both models are now loaded. GPT-2 has 124M parameters; Qwen3-0.6B has roughly 600M. If the previous cell took some time, that was mainly due to model download. The models are cached locally, so future runs will be much faster.

Next, we will generate text from both models using the same prompt. For GPT-2, we pass the raw text directly. For Qwen, we first format the prompt as a chat message using `apply_chat_template()`, then generate. Both use **top-p sampling** so the outputs are varied and natural.

In [51]:
# Generate text from both models using the same prompt.
prompt = "What is 2+2?"

# --- GPT-2: simple text completion ---
# GPT-2 is not instruction-tuned, so we pass the raw prompt directly
gpt2_inputs = gpt2_tokenizer(prompt, return_tensors="pt")

gpt2_output = gpt2_model.generate(
    **gpt2_inputs,
    max_new_tokens=64,
    do_sample=True,      # enable sampling for top-p
    top_p=0.9,           # nucleus sampling — sample from top 90% probability mass
    temperature=1.0,     # keep the distribution shape as-is
)

print("=== GPT-2 ===")
print(gpt2_tokenizer.decode(gpt2_output[0], skip_special_tokens=True))


# --- Qwen3-0.6B: instruction-tuned with chat template ---
# Qwen is a chat model — it expects messages in a structured format, not raw text
# apply_chat_template() formats the prompt into the special tokens Qwen was trained on
messages = [{"role": "user", "content": prompt}]

# tokenize=False returns the formatted string so we can inspect it; then we tokenize separately
chat_prompt = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,          # return formatted string, not tensor yet
    add_generation_prompt=True  # appends the assistant turn opener so model knows to respond
)

qwen_inputs = qwen_tokenizer(chat_prompt, return_tensors="pt").to(qwen_model.device)

qwen_output = qwen_model.generate(
    **qwen_inputs,
    max_new_tokens=100,
    do_sample=True,      # enable sampling for top-p
    top_p=0.9,           # nucleus sampling
    temperature=0.7,     # slightly lower than GPT-2 — Qwen benefits from a bit more focus
)

print("\n=== Qwen3-0.6B ===")
# decode only the newly generated tokens by slicing off the input length
print(qwen_tokenizer.decode(qwen_output[0][qwen_inputs.input_ids.shape[1]:], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== GPT-2 ===
What is 2+2? (10): What is 3+2? (11): What is 4+4? (12): What is 5+5? (13): What is 6+6? (14): What is 7+7? (15): What is 8+8? (16): What is 9+9? (

=== Qwen3-0.6B ===
<think>
Okay, so the question is "What is 2+2?" and I need to figure out the answer. Let me think. Well, the user is probably expecting a straightforward answer. But wait, sometimes there are different ways to interpret this. For example, in some contexts, adding two 2s might lead to a different result. Let me break it down.

First, in basic arithmetic, 2 plus 2 equals 4. That's simple. But maybe the question


# 5. (Optional) A Small Interactive LLM Playground
This section is optional. You do not need to implement it to complete the project. It is meant purely for exploration and will not significantly affect your core AI engineering skills.

If you are curious, you can build a simple interactive playground to experiment with text generation. You can:
- Create input widgets for the prompt, model selection, decoding strategy, and temperature
- Use Hugging Face's generate method to produce text based on the selected settings
- Display the model's response directly in the notebook output

You may find following links helpful:
- https://ipywidgets.readthedocs.io/en/latest/
- https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html

In [52]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# ── Step 1: Load models and tokenizers (reuse if already loaded above) ──────────

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")

qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
qwen_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.float16,  # halve memory usage vs float32
    device_map="auto",          # use GPU if available, else CPU
)

# ── Step 2: Helper function to generate text ────────────────────────────────────

def generate_text(prompt, model_name, strategy, temperature, max_new_tokens=128):
    # pick the right model and tokenizer based on selection
    if model_name == "GPT-2":
        tokenizer = gpt2_tokenizer
        model = gpt2_model
        # GPT-2 is not chat-tuned — pass raw prompt directly
        inputs = tokenizer(prompt, return_tensors="pt")
    else:
        tokenizer = qwen_tokenizer
        model = qwen_model
        # Qwen is chat-tuned — wrap prompt in chat template before tokenizing
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True  # appends assistant turn opener
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    # build generation kwargs based on chosen decoding strategy
    if strategy == "Greedy":
        strategy_kwargs = {"do_sample": False}
    elif strategy == "Top-p":
        strategy_kwargs = {"do_sample": True, "top_p": 0.9, "temperature": temperature}
    elif strategy == "Top-k":
        strategy_kwargs = {"do_sample": True, "top_k": 50, "temperature": temperature}

    # generate output token IDs
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        **strategy_kwargs,
    )

    # for Qwen, slice off the input tokens so we only decode the new response
    if model_name == "Qwen3-0.6B":
        output_ids = output_ids[:, inputs.input_ids.shape[1]:]

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# ── Step 3: Create UI elements ──────────────────────────────────────────────────

prompt_box = widgets.Textarea(
    value="What is 2+2?",
    description="Prompt:",
    layout=widgets.Layout(width="600px", height="80px")
)

model_selector = widgets.Dropdown(
    options=["GPT-2", "Qwen3-0.6B"],
    value="GPT-2",
    description="Model:",
)

strategy_selector = widgets.Dropdown(
    options=["Greedy", "Top-p", "Top-k"],
    value="Top-p",
    description="Strategy:",
)

temperature_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=2.0,
    step=0.1,
    description="Temperature:",
    style={"description_width": "initial"},  # prevent label from getting clipped
)

max_tokens_slider = widgets.IntSlider(
    value=128,
    min=16,
    max=256,
    step=16,
    description="Max tokens:",
    style={"description_width": "initial"},
)

# ── Step 4: Add a button to trigger generation ──────────────────────────────────

generate_button = widgets.Button(
    description="Generate",
    button_style="primary",   # blue button
)

output_area = widgets.Output()  # area where generated text will be printed

# ── Step 5: Define button behavior ─────────────────────────────────────────────

def on_generate_clicked(b):
    # clear previous output before showing new result
    output_area.clear_output()
    with output_area:
        print("Generating...")
        result = generate_text(
            prompt=prompt_box.value,
            model_name=model_selector.value,
            strategy=strategy_selector.value,
            temperature=temperature_slider.value,
            max_new_tokens=max_tokens_slider.value,
        )
        display(Markdown(f"**Output:**\n\n{result}"))

# bind the click event to the handler function
generate_button.on_click(on_generate_clicked)

# ── Step 6: Display the full UI ─────────────────────────────────────────────────

display(
    widgets.VBox([
        widgets.HTML("<h3>🧠 LLM Playground</h3>"),
        prompt_box,
        widgets.HBox([model_selector, strategy_selector]),   # side by side
        widgets.HBox([temperature_slider, max_tokens_slider]),
        generate_button,
        output_area,
    ])
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


# 6: Inference Engines: Ollama, vLLM, SGLang

So far, we loaded models directly in Python using HuggingFace's `transformers` library. This is great for learning, but in practice models run as **servers** that expose an API. Client applications send requests and receive responses over HTTP — the model itself stays loaded in memory (and on the GPU) between requests.

An **inference engine** handles all the heavy lifting: model loading, GPU memory management, request batching, and serving an HTTP API. Popular inference engines include:

| Engine | Best for |
|--------|----------|
| **Ollama** | Easy local use and experimentation |
| **vLLM** | High-throughput production serving |
| **SGLang** | Fast serving + structured output |

Most inference engines expose an **OpenAI-compatible API**. This means you can learn one client library (the `openai` Python package) and swap backends freely: Ollama for local development, vLLM or SGLang for production.

In future weeks, we'll learn about Ollama, set it up, and use it to easily load and build on top of modern powerful LLMs!

## 🎉 Congratulations!

You've just explored the internals of a real **LLM**. In this project you:
* Learned how **tokenization** works — from word-level to BPE — and why it matters
* Used `tiktoken` to compare tokenizers across different model generations
* Loaded GPT-2 and inspected its Transformer blocks and layers
* **Counted parameters** and understood where a model's capacity lives
* Learned how the model produces **logits and probabilities** to predict the next token
* Explored **decoding strategies**: greedy decoding and top-p (nucleus) sampling
* Witnessed the leap from GPT-2 (simple text completion) to Qwen3-0.6B — a modern model that **understands questions and thinks before answering**
* Learned about **inference engines** (Ollama, vLLM, SGLang) and the OpenAI-compatible API pattern

👏 **Great job!** Take a moment to celebrate. You now have a working mental model of how LLMs work — from raw text input all the way to generated output. The skills and intuitions you built here will serve as the foundation for everything that comes next.

